# Module 3: Online Evaluations

> Self-directed module, ~25 min.

Online evaluations score traces *as they land in your project*. We define an LLM-as-judge prompt + output schema, register it as a **run rule** against the project, and then generate new traces — each one is auto-scored within seconds.

Workflow: define the judge → create the rule → generate scored traces → query the scored runs back.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

from utils.langsmith_rules import create_run_rule, delete_run_rule


## Step 1 — Define the judge

Two pieces: a **prompt** that tells the judge what to look for, and a **JSON schema** that constrains the judge's output. LangSmith uses the schema to validate the judge response and to attach the scores as feedback on the run.

For the client research use case we score on three signals: it answered the question, it cited a source for any market/news claims, and it didn't hallucinate a client identity (only `acme-pension`, `smith-family-office`, `techcorp-treasury` are real).


In [ ]:
judge_prompt = (
    "You score a client research assistant's response.\n"
    "\n"
    "Mark correctness=true only if ALL of the following hold:\n"
    "  1. The response answers the user's actual question.\n"
    "  2. Any market or company news claims include at least one source URL.\n"
    "  3. The response does not invent client identities. Only 'acme-pension', 'smith-family-office', "
    "and 'techcorp-treasury' are real clients; an unknown client should be reported as not found, not fabricated.\n"
    "\n"
    "Otherwise mark correctness=false. Give one short sentence of comment either way."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the client research response is grounded, sourced, and honest about unknown clients.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if all three criteria are met"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}


## Step 2 — Create the run rule

`create_run_rule` registers the judge against the project. `sampling_rate=1.0` scores every run; the filter `eq(is_root, true)` keeps it from re-scoring every child LLM / tool span.

The printed URL opens the rule in the LangSmith UI — useful for inspecting, editing the prompt, or pausing the rule without redeploying the SDK call.

<!-- Screenshot placeholder: LangSmith UI rule editor showing the judge prompt + schema. -->


In [ ]:
online_rule = create_run_rule(
    client,
    project_name=PROJECT_NAME,
    display_name="client-research-online-correctness",
    sampling_rate=1.0,
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:    ", online_rule["id"])
print("Open in UI: ", online_rule["url"])


## Step 3 — Generate scored traces

Trigger a few more runs. Each one lands in the project, the rule fires, and a `correctness` feedback score is attached within ~30 seconds.


In [ ]:
trigger_prompts = [
    "Summarize Microsoft's most recent earnings. One paragraph.",
    "What's the risk profile of acme-pension?",
    "Look up the client 'unknown-fund' and tell me about their holdings.",
    "Research recent news on the energy sector. Search at most once.",
]

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

print(f"\nTotal: {total:.1f}s")
print(f"\nFeedback should appear in {online_rule['url']} within ~30 seconds.")


## Step 4 — Query scored runs

Once the rule has run, the feedback is attached to each scored trace. Pull them back with a feedback filter — both the runs that passed and the runs that failed are addressable individually.


In [ ]:
since = datetime.now(timezone.utc) - timedelta(hours=1)

scored_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    filter='eq(feedback_key, "correctness")',
    limit=20,
))
print(f"{len(scored_runs)} scored root run(s) in the last hour")
for r in scored_runs[:5]:
    fb = {f.key: f.score for f in (r.feedback_stats or {}).values()} if hasattr(r, 'feedback_stats') else {}
    print(f"  {r.id}  {r.name:25s}  feedback={fb}")

# Failed only (correctness < 0.5)
failed_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=since,
    is_root=True,
    filter='and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))',
    limit=20,
))
print(f"\n{len(failed_runs)} failed run(s) (correctness < 0.5)")


## Cleanup

Leave the rule active — Module 4 (annotation queues) routes runs based on its feedback. To remove it later:

```python
delete_run_rule(client, online_rule["id"])
```


## Recap

| Topic | API |
|---|---|
| Define LLM-as-judge | prompt string + JSON Schema for the judge output |
| Register the rule | `create_run_rule(client, project_name=..., llm_judge_prompt=..., llm_judge_schema=...)` |
| Query scored runs | `client.list_runs(filter='eq(feedback_key, "correctness")')` |
| Failed runs only | `filter='and(eq(feedback_key, ...), lt(feedback_score, 0.5))'` |

Next: `04_annotation_queues.ipynb` — route the failed runs to a human reviewer.
